In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import re

from tensorflow.keras.layers import Dense, GRU, Input, Dropout, Embedding
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.text import Tokenizer, text_to_word_sequence
from tensorflow.keras.preprocessing.sequence import pad_sequences

with open('train_data_true', 'r', encoding='utf-8') as f:
    texts_true = f.readlines()
    texts_true[0] = texts_true[0].replace('\ufeff', '') #убираем первый невидимый символ

with open('train_data_false', 'r', encoding='utf-8') as f:
    texts_false = f.readlines()
    texts_false[0] = texts_false[0].replace('\ufeff', '') #убираем первый невидимый символ

texts = texts_true + texts_false
count_true = len(texts_true)
count_false = len(texts_false)
total_lines = count_true + count_false
print(count_true, count_false, total_lines)


maxWordsCount = 1000
tokenizer = Tokenizer(num_words=maxWordsCount, filters='!–"—#$%&amp;()*+,-./:;<=>?@[\\]^_`{|}~\t\n\r«»', lower=True, split=' ', char_level=False)
tokenizer.fit_on_texts(texts)

dist = list(tokenizer.word_counts.items())
print(dist[:10])
print(texts[0][:100])


max_text_len = 10
data = tokenizer.texts_to_sequences(texts)
data_pad = pad_sequences(data, maxlen=max_text_len)
print(data_pad)

print( list(tokenizer.word_index.items()) )


X = data_pad
Y = np.array([[1, 0]]*count_true + [[0, 1]]*count_false)
print(X.shape, Y.shape)

indeces = np.random.choice(X.shape[0], size=X.shape[0], replace=False)
X = X[indeces]
Y = Y[indeces]


model = Sequential()
model.add(Embedding(maxWordsCount, 128, input_length = max_text_len))
model.add(GRU(128, return_sequences=True))
model.add(GRU(64))
model.add(Dense(2, activation='softmax'))
model.summary()

model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer=Adam(0.0001))

history = model.fit(X, Y, batch_size=32, epochs=50)

reverse_word_map = dict(map(reversed, tokenizer.word_index.items()))

def sequence_to_text(list_of_indices):
    words = [reverse_word_map.get(letter) for letter in list_of_indices]
    return(words)

t = "Это шедевр изобразительного".lower()
data = tokenizer.texts_to_sequences([t])
data_pad = pad_sequences(data, maxlen=max_text_len)
print( sequence_to_text(data[0]) )

res = model.predict(data_pad)
print(res, np.argmax(res), sep='\n')

## Izah

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
# TensorFlow-un artıq xəbərdarlıq mesajlarını gizlədir.
# DİQQƏT: TensorFlow import olunmazdan ƏVVƏL yazılmalıdır.


import numpy as np
# NumPy — massivlərlə işləmək üçün.
# Bu kodda:
# - np.array → siyahını massivə çevirmək
# - np.random.choice → nümunələri qarışdırmaq
# - np.argmax → ən yüksək ehtimallı sinfin nömrəsini tapmaq


import re
# re — adi ifadələr (regex) modulu.
# DİQQƏT: burada import olunub, amma AŞAĞIDA İSTİFADƏ OLUNMUR.
# Durğu işarələrini Tokenizer-in "filters" parametri təmizləyir.


from tensorflow.keras.layers import Dense, GRU, Input, Dropout, Embedding
# Keras qatları:
#
# Dense     → tam bağlı qat (çıxış qatı kimi).
# GRU       → rekurrent qat (Gated Recurrent Unit). LSTM-in sadələşdirilmiş
#             variantıdır: içində 2 qapı var (LSTM-də 3 qapı + ayrıca
#             yaddaş hüceyrəsi var). Nəticədə daha az parametr və daha
#             sürətli öyrənmə, keyfiyyət isə çox vaxt LSTM-ə yaxın olur.
# Input     → burada import olunub, amma İSTİFADƏ OLUNMUR
#             (giriş forması Embedding-in input_length ilə verilir).
# Dropout   → import olunub, amma İSTİFADƏ OLUNMUR
#             (overfitting-ə qarşı əlavə etmək olardı).
# Embedding → hər söz nömrəsini sıx vektora çevirir.


from tensorflow.keras.models import Sequential
# Sequential model: qatlar ardıcıl düzülür.


from tensorflow.keras.optimizers import Adam
# Adam optimizatoru. Burada onu obyekt kimi import edirik ki,
# öyrənmə sürətini (learning rate) özümüz təyin edə bilək.


from tensorflow.keras.preprocessing.text import Tokenizer, text_to_word_sequence
# Tokenizer → mətni rəqəmlərə çevirir.
# text_to_word_sequence → import olunub, amma İSTİFADƏ OLUNMUR.


from tensorflow.keras.preprocessing.sequence import pad_sequences
# pad_sequences → fərqli uzunluqlu söz-nömrə siyahılarını eyni
# uzunluğa gətirir (qısanı sıfırlarla doldurur, uzunu kəsir).
# GRU-ya verilən bütün nümunələr eyni uzunluqda olmalıdır.


# =====================================================================
# BU KOD NƏDİR? — SENTIMENT ANALİZİ (GRU ilə)
# =====================================================================
#
# Məqsəd: verilən cümləyə baxıb onun MÜSBƏT yoxsa MƏNFİ olduğunu demək.
# Bu, 2 sinifli təsnifat (classification) məsələsidir.
#
# ƏVVƏLKİ (LSTM) KODLA MÜQAYİSƏ:
# Kodun hər şeyi EYNİDİR, yalnız 2 yerdə dəyişiklik var:
#   1) import:   LSTM  →  GRU
#   2) modeldə:  LSTM qatları  →  GRU qatları
#   (və yoxlama cümləsi başqadır)
#
#                   LSTM                     GRU
# Qapılar:          3 qapı (input, forget,   2 qapı (update, reset)
#                   output)
# Yaddaş:           2 vəziyyət (h və c)      1 vəziyyət (yalnız h)
# Parametr sayı:    daha çox                 təxminən 15% az
# Sürət:            daha yavaş               daha sürətli
# Kiçik datada:     bəzən overfitting        çox vaxt daha yaxşı/sabit
#
# Praktikada hansının daha yaxşı olduğunu əvvəlcədən demək olmur,
# hər ikisini sınayıb nəticəyə baxmaq lazımdır.


with open('train_data_true', 'r', encoding='utf-8') as f:
    texts_true = f.readlines()
    texts_true[0] = texts_true[0].replace('\ufeff', '')  # убираем первый невидимый символ
# 'train_data_true' faylını oxuyur (MÜSBƏT cümlələr, hər sətirdə bir cümlə).
# Fayl kodla eyni qovluqda olmalıdır.
#
# readlines() → faylı sətir-sətir siyahıya çevirir.
# Hər element bir cümlədir, sonunda '\n' simvolu qalır
# (Tokenizer-in filters-i onu təmizləyəcək).
#
# '\ufeff' → görünməyən BOM simvolu. Yalnız faylın ƏVVƏLİNDƏ olur,
# ona görə yalnız ilk sətirdə [0] təmizləyirik.

with open('train_data_false', 'r', encoding='utf-8') as f:
    texts_false = f.readlines()
    texts_false[0] = texts_false[0].replace('\ufeff', '')  # убираем первый невидимый символ
# Eyni əməliyyat, amma MƏNFİ cümlələr üçün.

texts = texts_true + texts_false
# İki siyahını birləşdirir: əvvəl bütün müsbətlər, sonra bütün mənfilər.
# Bu sıranı yadda saxlayın: aşağıda etiketlər də eyni sırada yaradılır.

count_true = len(texts_true)
count_false = len(texts_false)
total_lines = count_true + count_false
print(count_true, count_false, total_lines)
# Neçə müsbət, neçə mənfi və cəmi neçə cümlə olduğunu çap edir.
# Məsələn: 50 50 100. İdeal halda iki say bir-birinə yaxın olmalıdır
# (balanslı data), yoxsa model çox olan sinfə meyilli olar.


maxWordsCount = 1000
# Lüğətin maksimum ölçüsü: ən çox işlənən 1000 söz nəzərə alınır.
# Qalan (nadir) sözlər atılır.
# Bu rəqəm Embedding qatının giriş ölçüsüdür.

tokenizer = Tokenizer(num_words=maxWordsCount, filters='!–"—#$%&amp;()*+,-./:;<=>?@[\\]^_`{|}~\t\n\r«»',
                      lower=True, split=' ', char_level=False)
# Söz səviyyəsində Tokenizer.
#
# num_words=1000 → ən çox işlənən 1000 sözü saxla.
#
# filters='...' → mətndən silinəcək simvollar (durğu işarələri,
#   xüsusi simvollar, sətir keçidi '\n', tire, dırnaqlar və s.).
#   DİQQƏT: '&amp;' HTML-dən qalıb. Bu o deməkdir ki, '&', 'a', 'm',
#   'p', ';' simvolları da filtrə düşür. Rus mətnində problem yoxdur,
#   amma latın hərfli mətndə 'a', 'm', 'p' silinərdi.
#   Düzgün variant: '&' yazmaq.
#
# lower=True → bütün sözləri kiçik hərfə çevirir
#   ("Позитив" və "позитив" eyni söz sayılır).
#
# split=' ' → sözləri boşluğa görə ayırır.
#
# char_level=False → HƏRF yox, SÖZ səviyyəsində işlə.

tokenizer.fit_on_texts(texts)
# Bütün cümlələri oxuyub hər sözə nömrə verir.
# Ən çox işlənən söz → 1, sonrakı → 2 və s.
# 0 nömrəsi boş saxlanılır (padding üçün).
# texts (cümlələr siyahısı) verilir, hər element ayrıca cümlə kimi işlənir.

dist = list(tokenizer.word_counts.items())
print(dist[:10])
# word_counts → hər sözün neçə dəfə işləndiyi lüğət.
# İlk 10 cütü çap edir. Sıralama tezliyə görə DEYİL,
# mətndə ilk rast gəlmə ardıcıllığına görədir.

print(texts[0][:100])
# İlk cümlənin ilk 100 simvolunu çap edir.
# Faylın düzgün oxunduğunu yoxlamaq üçündür.


max_text_len = 10
# Hər cümlə 10 sözə gətiriləcək.
# 10-dan uzun cümlə kəsilir, 10-dan qısa cümlə sıfırlarla doldurulur.
# Real tapşırıqlarda bu rəqəm adətən daha böyük olur (məsələn, 100-200).

data = tokenizer.texts_to_sequences(texts)
# Hər cümləni söz nömrələrinin siyahısına çevirir.
# Nəticə: siyahı içində siyahılar, məsələn [[45, 12, 7], [3, 88, 5, 9], ...]
# Cümlələrin uzunluğu fərqlidir.
#
# DİQQƏT: nömrəsi 1000 və daha böyük olan (nadir) sözlər
# nəticəyə düşmür, sadəcə atılır.

data_pad = pad_sequences(data, maxlen=max_text_len)
print(data_pad)
# Bütün cümlələri eyni uzunluğa (10) gətirir.
#
# Məsələn, maxlen=10:
#   [45, 12, 7]  →  [0, 0, 0, 0, 0, 0, 0, 45, 12, 7]
# Sıfırlar BAŞA əlavə olunur (default: padding='pre').
#
# Uzun cümlədə isə əvvəlki sözlər kəsilir, SON 10 söz qalır
# (default: truncating='pre').
#
# Forma: (cümlə_sayı, 10). Hamısı eyni ölçüdədir.

print(list(tokenizer.word_index.items()))
# word_index → söz → nömrə lüğəti (bütün sözlər, 1000-dən çox da ola bilər).
# Nömrələrin hansı sözə uyğun olduğunu yoxlamaq üçün çap edirik.


X = data_pad
# Giriş data: forma (cümlə_sayı, 10), yalnız nömrələr.

Y = np.array([[1, 0]] * count_true + [[0, 1]] * count_false)
# Etiketlər (düzgün cavablar), One-Hot formatında:
#   [1, 0] → MÜSBƏT
#   [0, 1] → MƏNFİ
#
# [[1, 0]] * count_true → [1, 0] cütünü count_true dəfə təkrarlayır.
# Sıra texts siyahısı ilə EYNİDİR: əvvəl müsbətlər, sonra mənfilər.
#
# Forma: (cümlə_sayı, 2).
# Softmax + categorical_crossentropy üçün One-Hot lazımdır.

print(X.shape, Y.shape)
# Formaları yoxlayır. Məsələn: (100, 10) (100, 2).
# Birinci rəqəm (nümunə sayı) hər ikisində eyni olmalıdır.

indeces = np.random.choice(X.shape[0], size=X.shape[0], replace=False)
# Nümunələri QARIŞDIRMAQ üçün təsadüfi sıra yaradır.
# replace=False → heç bir nömrə təkrarlanmır, yəni 0-dan N-ə qədər
# nömrələrin təsadüfi dəyişdirilmiş sırası alınır.
#
# Niyə qarışdırırıq? Data hələ "əvvəl bütün müsbətlər, sonra bütün
# mənfilər" sırasındadır. Qarışdırmasaq, model əvvəlcə yalnız müsbəti,
# sonra yalnız mənfini görər və öyrənmə pisləşər.

X = X[indeces]
Y = Y[indeces]
# X və Y EYNİ sıra ilə qarışdırılır, ona görə hər cümlə öz etiketi
# ilə qalır. Bu çox vacibdir: ayrı-ayrı qarışdırsaq, etiketlər pozulardı.


model = Sequential()

model.add(Embedding(maxWordsCount, 128, input_length=max_text_len))
# EMBEDDING QATI — hər söz nömrəsini 128 ədədlik sıx vektora çevirir.
# Vektor öyrənilir: mənaca yaxın sözlər zamanla yaxın vektorlar alır.
#
# Parametrlər:
#   1000 → lüğət ölçüsü
#   128  → hər sözün vektor ölçüsü
#   input_length=10 → giriş ardıcıllığının uzunluğu
#
# Forma: (None, 10) → (None, 10, 128)
# Parametr sayı: 1000 × 128 = 128000
#
# DİQQƏT: input_length arqumenti yeni Keras 3 versiyasında
# silinib, xəta verə bilər. Onda bu arqumenti çıxarıb əvvəldə
# model.add(Input((max_text_len,))) yazmaq lazımdır.

model.add(GRU(128, return_sequences=True))
# BİRİNCİ GRU qatı, 128 neyron.
#
# GRU-nun içində 2 qapı var:
#   - update gate (yeniləmə qapısı): köhnə məlumatın nə qədərini
#     saxlamaq, nə qədərini yeni məlumatla əvəz etmək lazımdır.
#   - reset gate (sıfırlama qapısı): keçmişin nə qədərini "unutmaq"
#     lazımdır.
# LSTM-dən fərqli olaraq ayrıca yaddaş hüceyrəsi (cell state) yoxdur,
# yalnız bir gizli vəziyyət (h) var.
#
# return_sequences=True → hər addımın (hər sözün) çıxışını qaytarır,
# yalnız sonuncunu yox. Bunun səbəbi: arxasında ikinci GRU var,
# ona isə tam ardıcıllıq lazımdır.
#
# Forma: (10, 128) → (10, 128)
# Parametr sayı (GRU-da 3 blok var: update, reset, yeni vəziyyət;
# Keras-ın defolt reset_after=True rejimində hər blokda 2 sürüşdürmə
# vektoru olur):
#   3 × (128 × 128 + 128 × 128 + 2 × 128) = 99072
# (LSTM-də bu qat 131584 parametrə malik idi.)

model.add(GRU(64))
# İKİNCİ GRU qatı, 64 neyron.
# return_sequences yoxdur (False), ona görə yalnız SON vəziyyəti
# qaytarır. Bu vektor bütün cümlənin "xülasəsi" kimidir.
#
# Forma: (10, 128) → (64,)
# Parametr sayı:
#   3 × (128 × 64 + 64 × 64 + 2 × 64) = 37248
# (LSTM-də bu qat 49408 idi.)

model.add(Dense(2, activation='softmax'))
# Çıxış qatı: 2 neyron (müsbət və mənfi).
# softmax → iki ehtimalın cəmi 1 olur.
# Məsələn: [0.92, 0.08] → 92% müsbət, 8% mənfi.
#
# Forma: (64,) → (2,)
# Parametr: 64 × 2 + 2 = 130

model.summary()
# Modelin quruluşunu çap edir.
# Ümumi parametr: 128000 + 99072 + 37248 + 130 = 264450.
# (LSTM versiyasında 309122 idi, yəni GRU təxminən 15% yüngüldür.)
# Rəqəmlər köhnə Keras versiyasında (reset_after=False) bir az fərqli
# çıxa bilər.

model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer=Adam(0.0001))
# loss → çoxsinifli təsnifat itkisi (One-Hot etiketlərlə işləyir).
# metrics → düzgün təxmin faizi.
# optimizer=Adam(0.0001) → öyrənmə sürəti 0.0001.
#   Defolt (0.001) -dən 10 dəfə kiçikdir: öyrənmə daha yavaş,
#   amma daha sabit olur. Kiçik datada bu, çəkilərin "atlamasının"
#   qarşısını alır.

history = model.fit(X, Y, batch_size=32, epochs=50)
# Öyrətmə: hər addımda 32 nümunə, bütün data 50 dəfə keçilir.
# history → hər epoch-un loss və accuracy tarixçəsi.
#
# DİQQƏT: validation_split yoxdur, yəni model yoxlama datası olmadan
# öyrədilir. Kiçik datada model əzbərləyə (overfitting) bilər:
# train accuracy 100% olar, amma yeni cümlələrdə zəif işləyər.
# Düzgün yoxlama üçün: model.fit(..., validation_split=0.2).


reverse_word_map = dict(map(reversed, tokenizer.word_index.items()))
# Tərs lüğət: nömrə → söz.
# word_index (söz → nömrə) cütlərinin hər birini reversed ilə
# çevirir: ('позитив', 45) → (45, 'позитив').

def sequence_to_text(list_of_indices):
    # Nömrələr siyahısını sözlər siyahısına çevirir.
    words = [reverse_word_map.get(letter) for letter in list_of_indices]
    # .get(nömrə) → nömrəyə uyğun sözü tapır.
    # (Dəyişənin adı "letter" qalıb əvvəlki hərf modelindən,
    # əslində söz nömrəsidir.)
    return (words)


t = "Это шедевр изобразительного".lower()
# Yoxlamaq istədiyimiz yeni cümlə: "Bu təsviri (sənətin) şedevridir".
# .lower() kiçik hərfə çevirir (Tokenizer də lower=True ilə bunu edir,
# amma əlavə etmək zərər vermir).
# DİQQƏT: cümlə yarımçıqdır, "изобразительного искусства" olmalı idi.
# "изобразительного" sözü lüğətdə yoxdursa, atılacaq və model yalnız
# "это шедевр" sözlərinə baxacaq.

data = tokenizer.texts_to_sequences([t])
# Cümləni nömrələr siyahısına çevirir. [t] → siyahı içində 1 cümlə.
# Nəticə: [[nömrə, nömrə, ...]]. Lüğətdə olmayan sözlər atılır.

data_pad = pad_sequences(data, maxlen=max_text_len)
# Cümləni öyrətmədəki kimi 10 uzunluğa gətirir (eyni qayda vacibdir).

print(sequence_to_text(data[0]))
# Modelin cümlədən HANSI sözləri tanıdığını göstərir.
# Lüğətdə olmayan söz bu siyahıda görünməz (atılıb).
# Əgər siyahı çox qısadırsa, model cümlənin çox hissəsini "görmür".

res = model.predict(data_pad)
# Modelin cavabı. Forma: (1, 2), məsələn [[0.93, 0.07]].
# 1-ci ədəd → müsbət ehtimalı, 2-ci ədəd → mənfi ehtimalı.

print(res, np.argmax(res), sep='\n')
# Əvvəl ehtimalları, sonra ən böyüyünün nömrəsini çap edir.
#   argmax = 0 → MÜSBƏT ([1, 0] etiketinə uyğun)
#   argmax = 1 → MƏNFİ  ([0, 1] etiketinə uyğun)
# sep='\n' → iki çıxışı ayrı sətirlərdə göstərir.